# Dokumentasi singkat notebook — Laboratorium forensik LLM (TokoBaju.id)

Notebook ini merupakan **runtime demonstrasi** untuk kode di `source_code_forensik.py`. Spesifikasi arsitektur, skema dataset, kontrak label forensik, dan daftar endpoint HTTP dirinci dalam **`DOKUMENTASI_PROYEK.md`** (unggah bersama skrip jika diperlukan salinan referensi di Colab).

## Ruang lingkup eksperimen

- **Target simulasi:** aplikasi *customer service* berbasis Flask yang memanggil model lokal melalui Ollama (default: `llama3`).
- **Korpus uji:** 28 skenario *prompt injection* dengan metadata `kategori` dan `kelas_ancaman` (stub taksonomi; pemetaan ke kerangka standar dilakukan secara manual di luar runtime).
- **Alur batch:** setiap sampel dikirim ke `/chat` → respons diklasifikasikan secara heuristik → agregat ditulis ke `laporan_forensik.json` → **ekspor dataset** ke `dataset/injection_runs.jsonl`, `dataset/injection_runs.csv`, dan `dataset/dataset_manifest.json`.

## Antarmuka publik lewat ngrok

Setelah tunnel aktif, UI tersedia pada path:

| Path | Fungsi |
|------|--------|
| `/` | Chat target dengan mitigasi tampilan teks (escape HTML di sisi klien). |
| `/dashboard` | Visualisasi agregat dan tabel per sampel. |
| `/dataset` | Indeks unduhan artefak dataset + pratinjau manifest. |
| `/exports/...` | Unduhan terkontrol (whitelist nama berkas). |

## Asumsi lingkungan Colab

Runtime Linux; proses `ollama serve` di belakang layar; koneksi keluar untuk unduhan model, paket Python, dan pembukaan sesi ngrok. **Kredensial ngrok tidak disimpan dalam repositori**; token dimasukkan melalui input interaktif di sel eksekusi (bukan tertulis di kode).

## Sel di bawah ini sebagai *infrastructure cells*

Kode pada sel-sel berikut persis mewakili perkakas build dan orkestrasi singkat; interpretasi hasil mengacu pada dokumentasi proyek dan berkas JSON/JSONL yang dihasilkan.

In [ ]:
# Instalasi lingkungan (Colab GPU opsional — Ollama pakai CPU juga boleh)
!apt-get update -qq && apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
%pip install -q ollama flask requests pyngrok

In [ ]:
# Jalankan daemon Ollama di background
import subprocess, time, os
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Ollama serve dimulai.")

In [ ]:
# Unduh model (butuh beberapa menit)
!ollama pull llama3

In [ ]:
# Unggah source_code_forensik.py dari laptop Anda
from google.colab import files
up = files.upload()
assert any(name.endswith("source_code_forensik.py") for name in up), "Unggah file source_code_forensik.py"
print("OK:", [k for k in up if k.endswith(".py")])

In [ ]:
# Orkestrasi: tunnel ngrok + batch 28 sampel + ekspor dataset
import importlib.util
from getpass import getpass
from IPython.display import HTML, display

path_py = "/content/source_code_forensik.py"
if not __import__("os").path.isfile(path_py):
    raise FileNotFoundError(f"Letakkan {path_py} (unggah di sel sebelumnya).")

spec = importlib.util.spec_from_file_location("forensik", path_py)
forensik = importlib.util.module_from_spec(spec)
spec.loader.exec_module(forensik)

token = getpass("NGROK_AUTHTOKEN (dashboard ngrok):")
if not token.strip():
    raise ValueError("Token kosong.")

public_url = forensik.jalankan_colab_dengan_ngrok(token.strip(), port=5001, jalankan_simulasi=True)

chat_url = public_url + "/"
dash_url = public_url + "/dataset"
board_url = public_url + "/dashboard"
display(HTML(f"""
<section style="font-family:system-ui;padding:18px;background:#12121c;color:#eaeaf0;border-radius:14px;border:1px solid #2a2a3a;max-width:640px">
  <h3 style="margin:0 0 14px;color:#ff6b4a">Sesi publik (ngrok)</h3>
  <p><strong>Chat</strong> — <a href="{chat_url}" target="_blank" style="color:#7ec8e3">{chat_url}</a></p>
  <p><strong>Dashboard forensik</strong> — <a href="{board_url}" target="_blank" style="color:#7ec8e3">{board_url}</a></p>
  <p><strong>Dataset &amp; unduhan JSONL/CSV</strong> — <a href="{dash_url}" target="_blank" style="color:#7ec8e3">{dash_url}</a></p>
  <p style="opacity:.75;font-size:.88rem;margin-top:12px">Batch menjalankan 28 skenario; artefak lokal: <code>hasil_serangan.json</code>, <code>laporan_forensik.json</code>, folder <code>dataset/</code>.</p>
</section>
"""))